<a href="https://colab.research.google.com/github/sponfork/Saki/blob/main/Tensor_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --force-reinstall --no-deps xformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

dataset = load_dataset("json", data_files="saki_training_dataset.jsonl", split="train")

def format_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in batch["messages"]
    ]
    return {"text": texts}

formatted_dataset = dataset.map(format_prompts, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="saki_7b_output",
        optim="adamw_8bit",
    ),
)

trainer.train()

In [ ]:
model.save_pretrained("saki_7b_mlbb_adapter")
tokenizer.save_pretrained("saki_7b_mlbb_adapter")
print("[✔] LoRA adapter saved to saki_7b_mlbb_adapter folder!")


In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "You are Saki, an elite Mobile Legends: Bang Bang (MLBB) draft strategy coach. Provide sharp, analytical counter-pick advice based on professional match data."
    },
    {
        "role": "user",
        "content": "The enemy team just picked Fanny and Joy for their core side. What heroes should we draft to counter them?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    use_cache=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== SAKI'S DRAFT ADVICE ===")
print(response)

In [ ]:
import json

with open("/content/mlbb_heroes_clean.json", "r", encoding="utf-8") as f:
    heroes_db = json.load(f)

print(f"[✔] Loaded {len(heroes_db)} heroes into heroes_db successfully!")

In [ ]:
import json
import gradio as gr

def generate_saki_advice(enemy_picks):
    if not enemy_picks:
        return "Select at least one enemy hero to generate draft counter analysis."

    matched_data = []
    for hero in heroes_db:
        if hero["name"] in enemy_picks:
            matched_data.append(hero)

    context_data = json.dumps(matched_data, indent=2)

    user_prompt = f"The enemy team locked in: {', '.join(enemy_picks)}. What are our best counter picks and strategy?"

    messages = [
        {
            "role": "system",
            "content": (
                "You are Saki, an elite Mobile Legends: Bang Bang (MLBB) draft strategy coach. "
                "Provide sharp, analytical counter-pick advice using strictly the statistical data context provided.\n\n"
                f"STATISTICAL DATA CONTEXT:\n{context_data}"
            ),
        },
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.7,
        use_cache=True,
    )

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

all_hero_names = sorted([hero["name"] for hero in heroes_db])

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏆 Saki: MLBB Live Ranked Draft Coach")
    gr.Markdown("Simulate an active draft room. Pick enemy heroes to get real-time counter advice from Saki.")

    with gr.Row():
        with gr.Column(scale=2):
            enemy_selection = gr.CheckboxGroup(
                choices=all_hero_names,
                label="🔴 Enemy Team Draft (Select Locked Heroes)",
                interactive=True
            )
            submit_btn = gr.Button("🧠 Ask Saki for Counter Advice", variant="primary")

        with gr.Column(scale=1):
            saki_output = gr.Textbox(
                label="💬 Saki's Tactical Analysis",
                lines=12,
                interactive=False
            )

    submit_btn.click(
        fn=generate_saki_advice,
        inputs=[enemy_selection],
        outputs=[saki_output]
    )

demo.launch(inline=False, share=True)